# Pre-train your first autoregressive Mixture-of-Experts protein language model

Companion notebook to the article. It builds and pre-trains an **MLA + sparse MoE**
decoder-only protein language model from scratch, end to end.

**What runs here**

1. Tokenize a protein FASTA into a memory-mapped `uint16` corpus
2. Pre-train the MoE model on the bundled demo data (~1 minute)
3. Plot the loss and sample sequences from the trained model
4. *Optional* — stream real UniRef50 data from the Hub and train a larger model

Code: [github.com/Dipayan26/MoE-Bind](https://github.com/Dipayan26/MoE-Bind) · Paper: [10.64898/2026.06.13.732043](https://doi.org/10.64898/2026.06.13.732043)

> **Runtime → Change runtime type → T4 GPU** is recommended but not required.
> Sections 1–3 run fine on CPU.

## 0 · Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU - CPU is fine for sections 1-3"

In [ ]:
!git clone -q https://github.com/Dipayan26/MoE-Bind.git
%cd MoE-Bind

### A note on `uv` in Colab

The repo uses [**uv**](https://docs.astral.sh/uv/). Locally you would run:

```bash
uv sync                                    # builds .venv/ from uv.lock
uv run python -m scripts.train --config ...
```

Colab is a special case. `uv sync` would build a **separate** `.venv/`, and this
notebook's Python cells (`import torch`, `from src...`) execute in Colab's own kernel,
which would not see that environment. It would also re-download a ~2.5 GB CUDA build of
torch that Colab already ships, matched to its driver.

So here we use uv as a fast installer into the *existing* environment — `uv pip install
--system` — and skip torch. Same resolver, no second interpreter.

In [ ]:
!pip install -q uv

# uv as a fast installer into Colab's own environment (torch is already present)
!uv pip install --system -q "transformers>=5.14" "datasets>=5.0" wandb pyyaml tqdm matplotlib

# Train offline - no W&B account needed. Swap for `!wandb login` to track properly.
%env WANDB_MODE=offline

All commands below are run as **modules** (`python -m scripts.<name>`) from the repo
root, because the code uses absolute `from src...` imports.

*Running this locally instead of in Colab? Use `uv sync` once, then prefix every command
below with `uv run` — e.g. `uv run python -m scripts.data_tokenize --config ...`.*

## 1 · Tokenize

Proteins use a **character-level tokenizer** — one token per amino acid, 31 tokens total
(20 canonical amino acids, 5 non-canonical, 6 special). No BPE, no merges.

The tokenizer streams the FASTA and writes a pre-allocated `uint16` memmap, so it scales
to corpora far larger than RAM. Train/val assignment is by hashing each sequence, which
makes the split deterministic regardless of file order.

In [ ]:
!cat configs/data_tokenize/demo_pretrain_tokenize.yaml

In [ ]:
!python -m scripts.data_tokenize --config configs/data_tokenize/demo_pretrain_tokenize.yaml

In [ ]:
# What the tokenizer actually did to a sequence
from src.data.tokenization.protein_character_tokenizer import ProteinTokenizerHF

tok = ProteinTokenizerHF()
seq = "MFLQSQKLWTMLLILAIWSPISHS"

print("vocab size :", tok.vocab_size)
print("sequence   :", seq)
print("tokens     :", tok._tokenize(seq)[:12], "...")
print("ids        :", tok(seq)["input_ids"][:12], "...")
print()
print("special tokens:", [t for t in tok.vocab if t.startswith("<")])

## 2 · Pre-train

The model is a decoder-only transformer where every block is:

```
x = x + MLA(RMSNorm(x))      # Multi-head Latent Attention  (compressed KV cache)
x = x + MoE(RMSNorm(x))      # sparse Mixture-of-Experts    (top-2 of N + shared)
```

We bump the demo to 300 iterations so the loss curve has something to show.

In [ ]:
import yaml

cfg = yaml.safe_load(open("configs/pretrain/demo/deepseek.yaml"))

cfg["training"]["max_iters"]  = 300
cfg["training"]["eval_iters"] = 25      # evaluate every 25 steps
cfg["training"]["warmup_steps"] = 20

yaml.dump(cfg, open("configs/pretrain/demo/deepseek_colab.yaml", "w"), sort_keys=False)
print(yaml.dump(cfg["model"], sort_keys=False))

In [ ]:
import subprocess, sys

# Capture stdout so we can plot the loss afterwards
proc = subprocess.run(
    [sys.executable, "-m", "scripts.train",
     "--config", "configs/pretrain/demo/deepseek_colab.yaml"],
    capture_output=True, text=True,
)
log = proc.stdout + proc.stderr
print("\n".join(l for l in log.splitlines()
                 if "Val Loss" in l or "parameters" in l or "FLOPs" in l or "Efficiency" in l))

### The loss curve

Two reference lines matter for a 31-token vocabulary:

- **ln(31) ≈ 3.43** — uniform random guessing
- **ln(20) ≈ 3.00** — knowing only that the 11 non-canonical tokens are rare

Anything between those two lines means the model has learned amino acid *frequencies*
and not much else. That is the expected outcome at this scale — see section 4 to fix it
with real data.

In [ ]:
import re, math
import matplotlib.pyplot as plt

pts = [(int(s), float(v)) for s, v in re.findall(r"step (\d+): Val Loss ([\d.]+)", log)]
steps, losses = zip(*pts)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(steps, losses, "o-", color="#4f46e5", lw=2, label="validation loss")
ax.axhline(math.log(31), ls="--", c="#9ca3af", label="ln(31) = uniform random")
ax.axhline(math.log(20), ls=":",  c="#ea580c", label="ln(20) = AA frequencies only")

ax.set_xlabel("step"); ax.set_ylabel("cross-entropy loss")
ax.set_title("MoE protein LM - demo pre-training")
ax.legend(); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

### Did the router collapse?

The failure mode unique to MoE is **router collapse** — a few experts win early and the
rest starve. Here we check expert usage directly by running a batch through the model
and counting how often each expert is selected.

Usage is reported as a share of all routing *slots*, so a perfectly balanced router gives
each expert `1 / n_experts` — 25% with four experts, regardless of top-k. Anything close
to zero for an expert means it has died.

In [ ]:
import torch, numpy as np
from src.models.config import DeepSeekConfig
from src.models.MOE_latent_bind import DeepSeekV3

ckpt = torch.load("checkpoints/demo/deepseek_pretrain/demo_deepseek_pretrain.pt",
                  map_location="cpu", weights_only=False)

model_cfg = DeepSeekConfig(**ckpt["config"]["model"])
model = DeepSeekV3(model_cfg)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# Feed one batch of real tokens and tap the router of every layer
data = np.memmap("data/demo/tokenized/demo_val.bin", dtype=np.uint16, mode="r")
x = torch.from_numpy(data[:model_cfg.block_size * 8].astype(np.int64)).view(8, -1)

usage = torch.zeros(model_cfg.n_experts)
with torch.no_grad():
    h = model.drop(model.wte(x))
    for block in model.h:
        h = h + block.attn(block.ln_1(h))
        flat = block.ln_2(h).view(-1, model_cfg.n_embd)
        logits = block.mlp.router(flat) + block.mlp.expert_bias
        _, idx = torch.topk(logits, model_cfg.n_experts_per_token, dim=-1)
        usage += torch.bincount(idx.flatten(), minlength=model_cfg.n_experts).float()
        h = h + block.mlp(block.ln_2(h))

share = (usage / usage.sum()).numpy()
for i, s in enumerate(share):
    print(f"expert {i}: {s:6.1%}  {'#' * int(s * 120)}")
print(f"\nperfectly balanced would be {1/model_cfg.n_experts:.1%} each")
print("expert_bias (load-balancing thermostat):", model.h[0].mlp.expert_bias.tolist())

## 3 · Sample from it

Give the model a starting methionine and let it continue. Generation is masked to the
20 canonical amino acids, so it cannot emit padding or delimiter tokens mid-sequence.

In [ ]:
from scripts.generate import generate

start = torch.tensor([[tok.vocab["M"]]])

for i in range(5):
    out = generate(model, start, max_new_tokens=70, temperature=0.9, top_k=10,
                   stop_token_ids=torch.tensor([tok.eos_token_id]))
    print(f"{i+1}. " + "".join(tok._convert_id_to_token(t) for t in out[0].tolist()))

These look protein-shaped — sensible letter frequencies, no absurd homopolymer runs —
and are biologically meaningless. A two-layer model trained on 180k tokens has learned
amino acid statistics and nothing more.

**Never judge a sampled model from one output.** Generate several, across several
prompts, before concluding anything.

## 4 · Optional — scale up on real UniRef50 data

*Requires a GPU runtime. Budget roughly 30–60 minutes.*

Chinchilla-style scaling suggests ~20 tokens per parameter, so a few-million-parameter
model wants tens of millions of tokens. We stream them from the Hub instead of
downloading the full corpus.

> **The `.shuffle()` below is not optional.** Most UniRef50 mirrors on the Hub are stored
> sorted by sequence length. Streaming the raw order gives a median length of 2,048
> residues with only ~3% of sequences inside the 40–512 window; shuffling brings the
> median to ~157 with ~90% inside it.

In [ ]:
from datasets import load_dataset
import os

os.makedirs("data/uniref", exist_ok=True)

N_SEQS = 100_000     # ~30M tokens; raise if you have runtime to spare

ds = (load_dataset("fredzzp/Uniref50", split="train", streaming=True)
      .shuffle(seed=42, buffer_size=10_000))

kept = 0
with open("data/uniref/uniref50_sample.fasta", "w") as f:
    for row in ds:
        seq = row["sequence"]
        if 40 <= len(seq) <= 512:
            f.write(f">seq{kept}\n{seq}\n")
            kept += 1
            if kept >= N_SEQS:
                break

print(f"wrote {kept:,} sequences")
!ls -lh data/uniref/uniref50_sample.fasta

In [ ]:
tok_cfg = yaml.safe_load(open("configs/data_tokenize/demo_pretrain_tokenize.yaml"))

tok_cfg["data"]["fasta_path"]       = "data/uniref/uniref50_sample.fasta"
tok_cfg["tokenizer"]["max_seq_len"] = 512
tok_cfg["tokenizer"]["total_tokens"] = 25_000_000
tok_cfg["output"]["train_bin"] = "data/uniref/tokenized/train.bin"
tok_cfg["output"]["val_bin"]   = "data/uniref/tokenized/val.bin"

yaml.dump(tok_cfg, open("configs/data_tokenize/uniref_colab.yaml", "w"), sort_keys=False)

!python -m scripts.data_tokenize --config configs/data_tokenize/uniref_colab.yaml

### A bigger model

6 layers, `n_embd: 256`, 8 experts top-2 plus a shared expert. Total parameters grow with
the expert count; **active** parameters per token do not.

In [ ]:
big = yaml.safe_load(open("configs/pretrain/demo/deepseek.yaml"))

big["model"].update(
    block_size=512, n_layer=6, n_head=8, n_embd=256,
    kv_lora_rank=32, q_lora_rank=32, rope_dim=32,
    n_experts=8, n_experts_per_token=2,
    expert_intermediate_size=384, shared_expert_intermediate_size=384,
)
big["data"].update(train_bin="data/uniref/tokenized/train.bin",
                   val_bin="data/uniref/tokenized/val.bin")
big["training"].update(
    max_iters=3000, warmup_steps=150, eval_iters=250,
    batch_size=16, gradient_accumulation_steps=4,
    learning_rate=6e-4, min_lr=6e-5,
    checkpoint_dir="checkpoints/colab/uniref_moe",
    out_mod_name="uniref_moe",
)
big["logging"]["id"] = "colab_uniref_moe"

yaml.dump(big, open("configs/pretrain/uniref_colab.yaml", "w"), sort_keys=False)

# Total vs active parameter accounting
c = DeepSeekConfig(**big["model"])
m = DeepSeekV3(c)
total  = sum(p.numel() for p in m.parameters())
routed = sum(p.numel() for n, p in m.named_parameters() if ".mlp.experts." in n)
active = total - routed + routed / c.n_experts * c.n_experts_per_token

print(f"total params        : {total:,}")
print(f"active per token    : {active:,.0f}")
print(f"ratio               : {total/active:.2f}x")
del m

In [ ]:
!python -m scripts.train --config configs/pretrain/uniref_colab.yaml

In [ ]:
# Sample from the scaled-up model
ckpt = torch.load("checkpoints/colab/uniref_moe/uniref_moe.pt",
                  map_location="cpu", weights_only=False)

big_model = DeepSeekV3(DeepSeekConfig(**ckpt["config"]["model"]))
big_model.load_state_dict(ckpt["model_state_dict"])
big_model.eval()

print(f"best val loss: {float(ckpt['best_val_loss']):.4f}   (ln(20) = 3.00)\n")

for i in range(5):
    out = generate(big_model, torch.tensor([[tok.vocab["M"]]]),
                   max_new_tokens=120, temperature=0.9, top_p=0.9,
                   stop_token_ids=torch.tensor([tok.eos_token_id]))
    print(f"{i+1}. " + "".join(tok._convert_id_to_token(t) for t in out[0].tolist()))

## Wrapping up

You have the full pre-training path: character-level tokenizer, low-rank attention, a
routed expert layer with a shared expert and a bias-based load balancer, a streaming
memmap data pipeline, and a training loop that runs on a laptop.

The obvious next step is more data and a bigger model. Section 4 above is the starting
point — 6 layers and 8 experts on 30M streamed tokens behaves very differently from the
two-layer demo, and it is where you can first check whether your own router is
specialising or collapsing.

- **Code:** [github.com/Dipayan26/MoE-Bind](https://github.com/Dipayan26/MoE-Bind)
- **Preprint:** *MoE-Bind: Guiding De Novo Protein Binder Generation with Sparse Experts*, [doi.org/10.64898/2026.06.13.732043](https://doi.org/10.64898/2026.06.13.732043)